## 🔹 Round 2 – Advanced Technical

**1.** Explain Spark’s **Tungsten** and **Catalyst Optimizer** and how they improve performance.

**2.** How would you handle **synchronization and avoid deadlocks** using threading or multiprocessing in Python?

**3.** Compare **Snowflake Schema and Star Schema** in data warehousing.

**4.** Explain how you would onboard a **Delta Lake catalog to Azure Synapse** for querying.

**5.** How do you capture **event logs and monitor performance on Databricks**?

**6.** What is the difference between **Presto and Spark’s distributed architecture**?

**7.** Can **Azure Synapse or Azure Data Explorer** work with near real-time streaming data sources? If yes, how?

## 1. Spark Catalyst Optimizer vs Tungsten

In Apache Spark, Catalyst Optimizer improves the execution plan, while Tungsten improves how data is processed in memory and how CPU resources are used. Both help Spark SQL and DataFrame workloads run faster.

### 1. Catalyst Optimizer

Catalyst is Spark SQL's query optimization framework. It transforms a logical query into an optimized physical execution plan.

How it works:

1. Parse the query: Convert SQL or DataFrame operations into an unresolved logical plan.

2. Analyze: Resolve table names, columns, and data types.

3. Optimize: Apply rules such as predicate pushdown, constant folding, and column pruning.

4. Plan execution: Select physical operators and strategies, such as broadcast hash join or sort-merge join.

5. Generate code where applicable: Work with Spark's execution engine to generate efficient code.

Example:

Python

Run

```
result = (
    df.filter(df.salary > 50000)
      .select("employee_id", "salary")
)
```

Catalyst can push filters toward the data source and read only the required columns when the source supports these optimizations.

### 2. Tungsten

Tungsten is a set of Spark execution-engine optimizations designed to improve CPU and memory efficiency.

Key techniques:

* Binary memory representation: Uses compact internal row formats, such as `UnsafeRow`, to reduce object overhead.

* Memory management: Improves memory efficiency and reduces garbage-collection pressure.

* Whole-stage code generation: Generates Java bytecode for compatible operator pipelines, reducing virtual function calls and intermediate overhead.

* CPU efficiency: Improves cache locality and reduces unnecessary object creation.

Example: A filter followed by a projection and aggregation may be compiled into a more efficient pipeline instead of processing each operator with substantial per-row overhead.

### 3. Key differences

| Feature       | Catalyst                                           | Tungsten                                        |
| ------------- | -------------------------------------------------- | ----------------------------------------------- |
| Primary focus | Query optimization                                 | Execution efficiency                            |
| Main layer    | Logical and physical planning                      | Memory and CPU execution                        |
| Techniques    | Predicate pushdown, column pruning, join selection | Binary rows, memory management, code generation |
| Main benefit  | Avoids unnecessary work                            | Executes remaining work more efficiently        |

### 4. How they improve performance together

SQL / DataFrame query

Catalyst Optimizer

Optimizes filters, columns, joins, and execution plans

Physical execution engine

Uses efficient memory layouts and code generation where supported

Faster Spark job

### 5. Interview-ready answer

“Catalyst is Spark SQL's query optimizer that improves logical and physical execution plans through techniques such as predicate pushdown, column pruning, and join optimization. Tungsten focuses on efficient execution using compact binary data representations, improved memory management, and whole-stage code generation. Together, they reduce unnecessary data processing, memory overhead, and CPU execution costs.”

Practical tip: For performance troubleshooting, inspect `df.explain("formatted")` to understand the physical plan, then use Spark UI to investigate shuffle volume, skew, spills, and task duration. Catalyst and Tungsten help, but they do not eliminate bottlenecks caused by poor partitioning or data skew.


## 2. How do you handle synchronization and avoid deadlocks in Python?

“Use synchronization mechanisms such as locks, semaphores, events, and queues to coordinate concurrent tasks and protect shared resources. To avoid deadlocks, I maintain a consistent lock acquisition order, minimize lock duration, and avoid unnecessary nested locks.”

### 1. Threading vs multiprocessing

| Feature         | Threading                              | Multiprocessing                                 |
| --------------- | -------------------------------------- | ----------------------------------------------- |
| Memory          | Shared memory                          | Separate process memory by default              |
| Best for        | I/O-bound tasks                        | CPU-bound tasks                                 |
| Synchronization | `threading.Lock`, `RLock`, `Semaphore` | `multiprocessing.Lock`, `Queue`, `Semaphore`    |
| Main risk       | Race conditions and deadlocks          | Deadlocks, IPC bottlenecks, resource contention |

### 2. Example: Prevent race conditions with a lock

Python

Run

```
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter

    for _ in range(10000):
        with lock:
            counter += 1

threads = [
    threading.Thread(target=increment)
    for _ in range(5)
]

for t in threads:
    t.start()

for t in threads:
    t.join()

print(counter)  # 50000
```

Why it works: The lock ensures only one thread updates the shared counter at a time. The `with` statement releases the lock automatically, including when an exception occurs.

### 3. How to avoid deadlocks

A deadlock occurs when tasks wait indefinitely for resources held by one another.

Common prevention techniques:

1. Consistent lock ordering: Always acquire multiple locks in the same order.

2. Minimize lock scope: Keep critical sections short.

3. Use timeouts: Use `lock.acquire(timeout=2)` to avoid waiting indefinitely.

4. Avoid nested locks: Reduce situations where a task holds one lock while waiting for another.

5. Use queues: Prefer `queue.Queue` or `multiprocessing.Queue` for producer-consumer workflows.

6. Release locks safely: Use context managers such as `with lock:`.

7. Avoid circular waits: Design resource dependencies so tasks cannot form a waiting cycle.

### 4. Example: Consistent lock ordering

Python

Run

```
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()

def update_resources():
    with lock_a:
        with lock_b:
            print("Resources updated safely")
```

All code that needs both locks must acquire `lock_a` before `lock_b`. This prevents circular waiting caused by inconsistent acquisition order.

### 5. When would you use multiprocessing?

For CPU-intensive tasks, I would use `multiprocessing` or `ProcessPoolExecutor` to execute work across multiple processes. When processes need to share state, I prefer message passing through queues where possible, rather than frequently modifying shared state.

Python

Run

```
from concurrent.futures import ProcessPoolExecutor

def square(n):
    return n * n

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(square, range(10)))

    print(results)
```

This example demonstrates parallel task execution without shared mutable state, reducing synchronization complexity.

### Strong interview conclusion

“I choose threading for I/O-bound operations and multiprocessing for CPU-bound work. I protect shared state with appropriate synchronization primitives, minimize critical sections, enforce consistent lock ordering, and prefer queues for communication. These practices reduce race conditions and help prevent deadlocks.”


## 3. Star Schema vs Snowflake Schema — Data Engineering Interview Answer

The main difference is that a Star Schema uses denormalized dimension tables, whereas a Snowflake Schema normalizes dimension tables into multiple related tables.

### 1. Architecture

![4 Database Schema Examples for Various Applications | Airbyte](https://images.openai.com/static-rsc-4/sypTcCcKkLpXZ09_4rwDYIx5OixhuWmr5257Nr3zZJURqh8PZfdDRnLC79MYCyFjKf9b4FEl-TqpZW3jwlTaoB0CKOvqFq6poPGZfpGqFwaIwoGrvYMs2yJyfAlYNTWd0PONwERVhC0q2HH_LyvWVH13RHUyZWz413gAkpGXEAk?purpose=inline)

Add to Favorites

Star Schema

One central fact table connected directly to dimension tables.

![Data Warehouses in Microsoft Fabric](https://images.openai.com/static-rsc-4/wxyvZelRCzSeqaHU75IBvXvPT8j3VCx3UuB5eYZNOaBXyPpnYiDWYwuCkJXnQSABq2Mj32PgStCEGmfdFEe3guE1q_xlDy-XYbL0oNfXgNL6ESHk4D3Z8e_ed40fdy38gwSaH4x1OgsdKSMAqulF7J-TIzjp8YBXfrkKiIFRh2E?purpose=inline)

Add to Favorites

Snowflake Schema

Dimension tables are split into related normalized tables.

### 2. Key differences

| Feature           | Star Schema                                  | Snowflake Schema                            |
| ----------------- | -------------------------------------------- | ------------------------------------------- |
| Dimension design  | Denormalized                                 | Normalized                                  |
| Table structure   | Simple and fewer tables                      | More tables and relationships               |
| Joins             | Fewer joins                                  | More joins                                  |
| Query performance | Often faster for BI queries                  | May require additional join processing      |
| Storage           | May use more storage for repeated attributes | Can reduce duplicated dimension data        |
| Data maintenance  | Simpler reporting model                      | More complex relationships to maintain      |
| Understandability | Easy for business users                      | More complex                                |
| Best use case     | Dashboards and analytical queries            | Complex hierarchies and reusable dimensions |

### 3. Example

Suppose you are building a retail data warehouse.

Star Schema

* `Fact_Sales`: product key, customer key, date key, sales amount.

* `Dim_Product`: product name, category, brand.

* `Dim_Customer`: customer name, city, state, country.

* `Dim_Date`: date, month, quarter, year.

All dimension tables connect directly to `Fact_Sales`.

Snowflake Schema

The product dimension is normalized into:

* `Dim_Product`: product key, product name, category key.

* `Dim_Category`: category key, category name, department key.

* `Dim_Department`: department key, department name.

This reduces repeated category and department attributes across product records but introduces additional joins.

### 4. Which would you choose?

* Star Schema: I prefer it for Power BI dashboards, sales reporting, and frequently executed analytical queries because it is simple and often provides efficient query performance.

* Snowflake Schema: I choose it when dimensions contain complex hierarchies, significant attribute duplication, or relationships that benefit from normalization.

Actual performance depends on the warehouse engine, data volume, indexing or clustering, and query patterns.

### 5. Interview-ready answer

“Star Schema has a central fact table connected to denormalized dimensions, making it simpler and often faster for analytical queries. Snowflake Schema normalizes dimensions into multiple related tables, reducing redundancy but increasing join complexity. For most BI and reporting workloads, I prefer Star Schema for simplicity and performance; I use Snowflake Schema when complex hierarchies and dimension maintainability justify the additional joins.”


## 4. How would you onboard a Delta Lake catalog to Azure Synapse?

“I would onboard Delta Lake data stored in Azure Data Lake Storage Gen2 to Azure Synapse by configuring secure storage access, exposing the Delta data through a supported query engine, and registering the required metadata for analytics.”

The exact approach depends on whether I use Synapse Spark pools or a Synapse serverless SQL pool.

## 1. High-level architecture

Delta Lake tables

ADLS Gen2 · Parquet files + Delta transaction log

Authentication and metadata setup

Managed identity · RBAC/ACLs · Catalog registration

Synapse Spark

Read Delta tables using Spark/Delta Lake support

Serverless SQL pool

Query supported Delta tables through SQL

Power BI / SQL analytics

## 2. Step-by-step implementation

### Step 1: Configure access to ADLS Gen2

* Identify the storage account, container, and Delta table paths.

* Configure Synapse workspace managed identity or another approved identity.

* Grant appropriate Azure RBAC permissions and ADLS Gen2 ACLs.

* Create a linked service or external data source where required.

Example Delta table path:

Python

Run

```
delta_path = (
    "abfss://curated@myaccount.dfs.core.windows.net/"
    "sales"
)
```

### Step 2: Query Delta Lake through Synapse Spark

If you use a Synapse Spark pool with compatible Delta Lake support, read the table directly:

Python

Run

```
df = spark.read.format("delta").load(delta_path)

df.createOrReplaceTempView("sales")

df.show()
```

For persistent catalog registration, use the supported metastore/catalog configuration for your Synapse environment rather than assuming a Databricks Unity Catalog can be attached directly.

### Step 3: Query Delta tables using serverless SQL

For supported Delta table layouts and Synapse configurations, you can expose a Delta path using a SQL view.

SQL

```
CREATE VIEW dbo.sales AS
SELECT *
FROM OPENROWSET(
    BULK 'https://myaccount.dfs.core.windows.net/curated/sales/',
    FORMAT = 'DELTA'
) AS rows;
```

This example assumes that the Synapse serverless SQL pool supports the Delta format used by the table and has the required storage permissions. Verify compatibility with your workspace's capabilities and Delta table features before production deployment.

### Step 4: Register metadata and validate

* Create or update the required database, schema, tables, or views.

* Validate column names, data types, row counts, and null values.

* Compare query results against the source Delta table.

* Test access controls and confirm that unauthorized users cannot access the data.

## 3. Important distinction: Delta catalog vs Delta files

A common interview follow-up is whether Azure Synapse can directly onboard an existing Databricks Unity Catalog.

These are different things:

* Delta Lake table: Data files plus the `_delta_log` transaction log in ADLS.

* Catalog/metastore: Metadata describing tables, schemas, permissions, and locations.

* Synapse SQL view: A SQL-accessible definition over supported data in storage.

Registering a path or creating a view does not automatically migrate Unity Catalog metadata, privileges, lineage, or governance.

## 4. Interview-ready conclusion

“I first configure secure access to ADLS Gen2, then choose the appropriate Synapse query engine. For Spark workloads, I read Delta tables using a compatible Delta Lake runtime. For serverless SQL, I expose supported Delta tables using `OPENROWSET` or SQL views. Finally, I validate schema, data consistency, permissions, and query performance. I also distinguish registering Delta data from migrating an entire external catalog such as Unity Catalog.”


## 5. How do you capture event logs and monitor performance on Databricks?

“I use Databricks Spark UI, cluster event logs, Spark event logs, cluster metrics, and centralized monitoring to troubleshoot jobs, identify performance bottlenecks, and track cluster utilization.”

### 1. How I capture event logs

A. Spark event logs

Spark event logs record events such as job starts, stage execution, task completion, and executor activity. They can be used to investigate completed Spark applications.

Python

Run

```
spark.conf.set("spark.eventLog.enabled", "true")
```

For persistent event logs, configure the Spark event-log directory before the Spark application starts, typically through cluster or compute configuration:

```
spark.eventLog.enabled true
spark.eventLog.dir abfss://logs@storage.dfs.core.windows.net/spark-events
```

The storage location must be supported by the configured Spark runtime and accessible to the application. In managed Databricks environments, use the supported compute configuration and permissions for your workspace.

B. Databricks cluster event logs

Cluster event logs capture lifecycle events such as cluster creation, startup, resizing, and termination. These help diagnose infrastructure and autoscaling issues.

C. Job and task logs

Capture notebook output, task errors, job run status, and stack traces to troubleshoot failed workflows.

### 2. How I monitor performance

| Tool                          | What I monitor                                                       |
| ----------------------------- | -------------------------------------------------------------------- |
| Spark UI                      | Jobs, stages, tasks, shuffle, spills, skew                           |
| Databricks job run details    | Execution duration, task failures, retries                           |
| Cluster metrics               | CPU, memory, utilization, worker scaling                             |
| Spark event logs              | Historical execution details and task-level behavior                 |
| System tables / audit logs    | Job history, compute usage, and platform activity where available    |
| Azure Monitor / Log Analytics | Centralized infrastructure and diagnostic monitoring when configured |

### 3. Example: Identify a slow Spark job

Suppose a Databricks ETL pipeline takes 90 minutes instead of its usual 30 minutes.

1. Open the Spark UI and identify the slowest stage.

2. Check task duration distribution to detect skewed partitions.

3. Review shuffle read/write and disk spill metrics.

4. Check executor CPU and memory utilization.

5. Inspect failed tasks, retries, and data-source read times.

6. Apply targeted fixes, such as broadcast joins, skew handling, partition tuning, or file pruning.

7. Compare execution metrics before and after optimization.

### 4. Capture custom performance metrics in PySpark

For basic application-level monitoring, record execution duration and output row counts.

Python

Run

```
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("etl_monitor")

start = time.perf_counter()

try:
    result = transformed_df
    row_count = result.count()

    logger.info(
        "ETL completed: rows=%s duration_seconds=%.2f",
        row_count,
        time.perf_counter() - start
    )

except Exception:
    logger.exception("ETL pipeline failed")
    raise
```

Production note: `count()` triggers a Spark action and may add significant processing overhead. Use it only when the metric is needed, or use existing pipeline metrics where appropriate.

### 5. Best practices

* Enable supported event logging and configure durable log retention.

* Monitor job duration, failure rate, data volume, and SLA breaches.

* Use alerts for failed jobs and abnormal execution times.

* Secure logs because they may contain sensitive information.

* Avoid logging credentials, tokens, or personally identifiable information.

* Correlate job runs with pipeline IDs and source data batches.

Strong interview conclusion: “I use Spark UI for immediate troubleshooting, event logs for historical analysis, Databricks job metrics for workflow monitoring, and Azure Monitor or Log Analytics for centralized observability. I correlate execution time, shuffle, spill, CPU, memory, and failure metrics to identify the root cause before changing cluster size or Spark configurations.”


## 6. Presto vs Apache Spark Distributed Architecture

Presto (now commonly associated with Trino's distributed SQL engine lineage) is designed primarily for fast, interactive SQL analytics, while Apache Spark is a general-purpose distributed processing engine supporting SQL, batch processing, streaming, and machine learning.

### 1. Architecture comparison

![Introduction to Presto: Open Source SQL Query Engine that's changing Big Data Analytics - DEV Community](https://images.openai.com/static-rsc-4/nrE20zJ07UJTrYOgsWFl09eJYQr_ru3_19H90DdxYICvqEiSv5lz24Bb7G3gplAY4m2gLu2mNs6Gx-V6MgOyU-rrMz_STuNRqXPt1kGiPgo0qbgwFOF5l0Tlpc65X24_9elOX9mP5f3SiqBtdS5NHdsi7yrlAdpUeKvh6kQTdjQ?purpose=inline)

Add to Favorites

Presto architecture

* Coordinator plans and schedules SQL queries.

* Worker nodes execute distributed tasks.

* Connectors read from sources such as Hive, object storage, and relational databases.

* Results are streamed back to the client.

![Demystifying the Internal Workings of Apache Spark Architecture | by Shubham Gupta | Medium](https://images.openai.com/static-rsc-4/PoEeidZJb_-RkvId6mUBVtTJDdn9QOXW_-Y-ooE_n3bwmYtKCQWU66sAzpn-o9eCXuthcPRPE98BwoBiax3R2q9l5ZevhHJULtqmYUE-ObuaPZCs3WvantwUyojmjTl_igvGyWIgqt5QRqNh_XtNU89uriyy6rpNpTyoAPYGw94?purpose=inline)

Add to Favorites

Spark architecture

* Driver builds the execution plan and coordinates execution.

* Cluster manager allocates resources.

* Executors process tasks across partitions.

* DAG scheduler divides jobs into stages separated by shuffle boundaries.

### 2. Key differences

| Feature           | Presto                                                        | Apache Spark                                                                  |
| ----------------- | ------------------------------------------------------------- | ----------------------------------------------------------------------------- |
| Primary purpose   | Interactive distributed SQL                                   | General-purpose distributed computing                                         |
| Main components   | Coordinator and workers                                       | Driver, executors, cluster manager                                            |
| Execution model   | Distributed SQL operators and pipelines                       | DAG of stages and tasks                                                       |
| Data processing   | Primarily query execution                                     | Batch, SQL, streaming, ML, and more                                           |
| Intermediate data | Often pipelined in memory; may spill                          | In-memory processing with disk spill and shuffle files                        |
| Fault tolerance   | Query/task-level recovery varies by version and configuration | Task retries, lineage-based recomputation, and checkpointing where applicable |
| Data sources      | Federated queries through connectors                          | Broad source connectors and processing libraries                              |
| Best use case     | Low-latency SQL analytics                                     | Complex ETL and large-scale transformations                                   |

### 3. How query execution differs

Presto:

1. The coordinator receives a SQL query.

2. It parses, optimizes, and creates a distributed execution plan.

3. Workers execute fragments in parallel.

4. Data flows between operators and workers, and results return to the client.

Spark:

1. The driver receives a SQL query or application.

2. Spark creates an execution plan and a DAG.

3. The DAG scheduler splits execution into stages.

4. Executors run tasks against data partitions.

5. Shuffle, caching, and intermediate storage are used as required.

### 4. Which would you choose?

* Presto: Interactive BI dashboards, ad hoc SQL, and federated queries across multiple data sources.

* Spark: Large-scale ETL, complex joins and aggregations, streaming pipelines, and ML feature engineering.

For example, in an Azure data platform, I might use Spark on Databricks to transform raw data into curated Delta Lake tables, then use a SQL engine such as Trino or a suitable warehouse engine for interactive analytics.

### Strong interview conclusion

“Presto is optimized for interactive, distributed SQL queries using a coordinator-worker architecture. Spark uses a driver-executor architecture and a DAG execution engine, making it suitable for broader workloads such as ETL, batch, streaming, and machine learning. I choose based on workload complexity, latency requirements, data sources, and operational cost.”


## 7. Can Azure Synapse or Azure Data Explorer handle near-real-time streaming?

Yes. Both support near-real-time analytics, but they serve different purposes.

* Azure Data Explorer (ADX): Best suited for high-volume telemetry, application logs, IoT events, and time-series analytics with low query latency.

* Azure Synapse Analytics: Supports streaming-oriented solutions through Spark Structured Streaming, Event Hubs integration, and streaming ingestion patterns. It is also useful for combining streaming data with historical warehouse analytics.

## 1. Architecture

Event producers

Applications · IoT devices · Transactions · Logs

Azure Event Hubs

Durable event ingestion and buffering

Azure Data Explorer

Streaming ingestion → KQL queries → dashboards

Azure Synapse

Spark streaming → transformations → analytics storage

Choose the processing and storage path based on latency, transformation, and analytical requirements.

## 2. Azure Data Explorer (ADX): Near-real-time ingestion

ADX is usually my first choice when events must become queryable very quickly. It supports continuous ingestion from Event Hubs, IoT Hub, and other supported sources.


Implementation steps:

1. Create an Event Hub and publish events from applications or IoT devices.

2. Create a database and target table in ADX.

3. Configure an Event Hubs data connection, preferably using managed identity.

4. Configure streaming ingestion if very low ingestion latency is required.

5. Query the ingested events using Kusto Query Language (KQL).

Example KQL query:

kusto

```
Telemetry
| where Timestamp > ago(5m)
| summarize EventCount = count()
    by bin(Timestamp, 1m), EventType
| order by Timestamp desc
```

This calculates event counts per minute over the last five minutes.

Important: ADX supports subsecond ingestion scenarios under suitable conditions, but actual end-to-end latency depends on configuration, workload, and query patterns. For sustained high-volume ingestion, queued ingestion may be more appropriate.



## 3. Azure Synapse: Near-real-time processing

Synapse is useful when streaming events need complex transformations or integration with historical lakehouse and warehouse data. A common architecture uses Event Hubs, Synapse Spark Structured Streaming, and ADLS Gen2.


Example using Spark Structured Streaming:

Python

Run

```
from pyspark.sql.functions import (
    col, from_json, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType
)

schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType())
])

events = (
    spark.readStream
    .format("eventhubs")
    .options(**event_hub_config)
    .load()
)

parsed = (
    events
    .select(
        from_json(
            col("body").cast("string"), schema
        ).alias("event")
    )
    .select("event.*")
    .withColumn("processed_at", current_timestamp())
)

query = (
    parsed.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .start(delta_path)
)
```

Here, `event_hub_config`, `checkpoint_path`, and `delta_path` represent your configured Event Hubs options, durable checkpoint location, and ADLS Gen2 Delta destination. The Event Hubs Spark connector must be available in the Synapse Spark environment.

Note: This is a streaming ingestion example, not a guarantee of subsecond end-to-end latency. Actual latency depends on trigger intervals, micro-batch processing, and storage behavior.

## 4. Which service would you choose?

| Requirement                                       | Recommended approach                                    |
| ------------------------------------------------- | ------------------------------------------------------- |
| Live telemetry dashboards and log analytics       | Azure Data Explorer                                     |
| Complex streaming transformations                 | Synapse Spark Structured Streaming                      |
| Historical data lake plus streaming processing    | Synapse Spark + ADLS Gen2 / Delta Lake                  |
| Low-latency queries over continuous event streams | ADX streaming ingestion                                 |
| Enterprise data warehouse reporting               | Synapse SQL, potentially alongside a streaming platform |

### Interview-ready conclusion

“Yes, both can work with near-real-time sources. I would use Event Hubs as the ingestion layer. For low-latency telemetry analytics, I would connect Event Hubs directly to Azure Data Explorer and query with KQL. For complex transformations and integration with historical data, I would use Synapse Spark Structured Streaming to process events and write curated data to ADLS Gen2 in Delta format. I would monitor ingestion lag, processing latency, failed records, and data freshness to meet the SLA.”
